### **Submissão 1B — Modelo PyTorch**

**Grupo 1 · MIA · Aprendizagem Profunda**

Modelo: Melhor DNN/LSTM PyTorch
Output: `subm1-g1-MIA-B.csv`

In [1]:
import numpy as np
import pandas as pd
import pickle
import sys, os, re
import torch
import torch.nn as nn

sys.path.append(os.path.abspath('../src'))
from utils import transform_new_texts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

print('\n1. A carregar modelo PyTorch...')
with open('../models/pytorch.pkl', 'rb') as f:
    meta = pickle.load(f)

transformers = meta['transformers']
class_names = meta['class_names']
model_class = meta['model_class']
input_size = meta['input_size']
num_classes = meta['num_classes']

print(f'   Modelo: {model_class}')
print(f'   Classes: {list(class_names)}')

Dispositivo: cuda

1. A carregar modelo PyTorch...
   Modelo: DNNWide
   Classes: [np.str_('Anthropic'), np.str_('Google'), np.str_('Human'), np.str_('Meta'), np.str_('OpenAI')]


In [2]:
print('2. A carregar dataset de submissão...')
df = pd.read_csv('../data/subm1.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()

textos = df['text'].tolist()
ids = df['id'].tolist()
print(f'   {len(textos)} textos carregados')

2. A carregar dataset de submissão...
   150 textos carregados


In [3]:
print('3. A preparar dados...')

# Determinar se o modelo usa features tabulares ou sequenciais
is_sequential = model_class in ['EmbeddingDNN', 'BiLSTM', 'BiLSTMGloVe']

if is_sequential:
    word2idx = meta['word2idx']
    max_seq_len = meta['max_seq_len']

    def simple_tokenize(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return [t for t in text.split() if len(t) > 1]

    def texts_to_sequences(texts, word2idx, max_len):
        sequences = []
        for text in texts:
            tokens = simple_tokenize(text)
            seq = [word2idx.get(t, 1) for t in tokens[:max_len]]
            seq = seq + [0] * (max_len - len(seq))
            sequences.append(seq)
        return np.array(sequences)

    X_seq = texts_to_sequences(textos, word2idx, max_seq_len)
    X_input = torch.LongTensor(X_seq).to(device)
    print(f'   {X_seq.shape[0]} textos → sequências {X_seq.shape}')
else:
    X = transform_new_texts(textos, transformers)
    X_input = torch.FloatTensor(X).to(device)
    print(f'   {X.shape[0]} textos → {X.shape[1]} features')

3. A preparar dados...
   150 textos → 3013 features


In [4]:
print('4. A classificar...')

# ── Todas as arquiteturas ──

class DNNWide(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

class DNNNarrow(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes))
    def forward(self, x): return self.net(x)

class EmbeddingDNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, num_classes))
    def forward(self, x):
        embedded = self.embedding(x)
        mask = (x != 0).unsqueeze(-1).float()
        pooled = (embedded * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.net(pooled)

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        combined = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(combined))

class BiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.gru(embedded)
        combined = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(combined))

class BiLSTMGloVe(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, freeze_embed=True):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix), freeze=freeze_embed, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes))
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        combined = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(combined))

# ── Reconstruir o modelo correto ──
if model_class == 'DNNWide':
    model = DNNWide(input_size, num_classes)
elif model_class == 'DNNNarrow':
    model = DNNNarrow(input_size, num_classes)
elif model_class == 'EmbeddingDNN':
    model = EmbeddingDNN(meta['vocab_size'], meta['embed_dim'], num_classes)
elif model_class == 'BiLSTM':
    model = BiLSTM(meta['vocab_size'], meta['embed_dim'], meta['hidden_dim'], num_classes)
elif model_class == 'BiGRU':
    model = BiGRU(meta['vocab_size'], meta['embed_dim'], meta['hidden_dim'], num_classes)
elif model_class == 'BiLSTMGloVe':
    model = BiLSTMGloVe(meta['embedding_matrix'], meta['hidden_dim'], num_classes,
                         freeze_embed=meta.get('freeze_embed', True))
else:
    raise ValueError(f'Modelo desconhecido: {model_class}')

model = model.to(device)
model.load_state_dict(torch.load('../models/pytorch.pth', map_location=device, weights_only=True))
model.eval()

with torch.no_grad():
    _, y_pred = torch.max(model(X_input), 1)
    y_pred = y_pred.cpu().numpy()

labels_pred = [class_names[i] for i in y_pred]
print(f'   ✅ {len(labels_pred)} previsões feitas')


4. A classificar...
   ✅ 150 previsões feitas


In [5]:
print('5. A exportar CSV...')

df_out = pd.DataFrame({
    'ID': ids,
    'Text': textos,
    'Labels': labels_pred
})

output_path = '../subm1/subm1-g1-MIA-B.csv'
os.makedirs('../subm1', exist_ok=True)
df_out.to_csv(output_path, sep=';', index=False, encoding='utf-8')

print(f'✅ Ficheiro guardado: {output_path}')
print(f'\nDistribuição das previsões:')
print(df_out['Labels'].value_counts().to_string())
print(f'\nPrimeiras 10 linhas:')
print(df_out.head(10).to_string(index=False))

5. A exportar CSV...
✅ Ficheiro guardado: ../subm1/subm1-g1-MIA-B.csv

Distribuição das previsões:
Labels
Human        62
Google       24
OpenAI       23
Meta         22
Anthropic    19

Primeiras 10 linhas:
   ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   